# WLASL2000 I3D — webcam / video quick test

This notebook lives in **jetson-code** but loads models and code from your **WLASL** checkout (`code/I3D`).

**Required on your machine (e.g. laptop):**
- A WLASL repo with `code/I3D/pytorch_i3d.py`, `videotransforms.py`, `weights/rgb_imagenet.pt`, `preprocess/wlasl_class_list.txt`, and `archived/asl2000/FINAL_nslt_2000_....pt`.
- **Set where WLASL lives** using the next cell (`WLASL_ROOT` env var **or** edit the path). No paths are hardcoded for a specific PC.
- Webcam: OpenCV. GPU recommended (`cuda`).

**Optional:** use the last cell to classify a short `.mp4` instead of the webcam.

In [ ]:
import os
from pathlib import Path

# How to point at WLASL (pick one):
#   A) export WLASL_ROOT=/path/to/WLASL   (folder that CONTAINS code/I3D)
#   B) Replace the placeholder Path in the else branch with your WLASL repo root
if os.environ.get("WLASL_ROOT"):
    WLASL_ROOT = Path(os.environ["WLASL_ROOT"]).expanduser().resolve()
else:
    WLASL_ROOT = Path("/__EDIT_THIS_PATH_TO_WLASL_REPO_ROOT__")

I3D_DIR = WLASL_ROOT / "code" / "I3D"
if not I3D_DIR.is_dir():
    raise FileNotFoundError(
        "Set environment variable WLASL_ROOT or assign WLASL_ROOT in this cell to your WLASL repo root "
        f"(expected {I3D_DIR.as_posix()}). Current WLASL_ROOT={WLASL_ROOT!r}"
    )

NUM_CLASSES = 2000
RGB_INIT = I3D_DIR / "weights" / "rgb_imagenet.pt"
FINAL_WEIGHTS = (
    I3D_DIR
    / "archived"
    / "asl2000"
    / "FINAL_nslt_2000_iters=5104_top1=32.48_top5=57.31_top10=66.31.pt"
)
CLASS_LIST = I3D_DIR / "preprocess" / "wlasl_class_list.txt"

WEBCAM_INDEX = 0
NUM_FRAMES = 64
TOP_K = 10

In [ ]:
import sys

import cv2
import numpy as np
import torch
import torch.nn as nn

sys.path.insert(0, str(I3D_DIR))
os.chdir(I3D_DIR)

import videotransforms
from pytorch_i3d import InceptionI3d

In [ ]:
def load_gloss_table(path: Path):
    idx_to_gloss = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t", 1)
            if len(parts) == 2:
                idx_to_gloss[int(parts[0])] = parts[1]
    return idx_to_gloss


idx_to_gloss = load_gloss_table(CLASS_LIST)
assert len(idx_to_gloss) >= NUM_CLASSES, "wlasl_class_list.txt missing or too short"

In [ ]:
def preprocess_frames_hwc(frames_np: np.ndarray) -> np.ndarray:
    """Match WLASL `datasets/nslt_dataset_all.load_rgb_frames_from_video` + CenterCrop(224)."""
    out = []
    for img in frames_np:
        w, h, c = img.shape
        if w < 226 or h < 226:
            d = 226.0 - min(w, h)
            sc = 1 + d / min(w, h)
            img = cv2.resize(img, dsize=(0, 0), fx=sc, fy=sc)
        img = (img / 255.0) * 2 - 1
        out.append(img)
    arr = np.asarray(out, dtype=np.float32)
    return videotransforms.CenterCrop(224)(arr)


def video_to_tensor(pic: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(pic.transpose([3, 0, 1, 2])).float()

In [ ]:
def load_i3d_wlasl2000(device: torch.device) -> nn.Module:
    if not RGB_INIT.is_file():
        raise FileNotFoundError(f"Missing {RGB_INIT}")
    if not FINAL_WEIGHTS.is_file():
        raise FileNotFoundError(f"Missing {FINAL_WEIGHTS}")

    i3d = InceptionI3d(400, in_channels=3)
    i3d.load_state_dict(torch.load(RGB_INIT, map_location="cpu"))
    i3d.replace_logits(NUM_CLASSES)
    sd = torch.load(FINAL_WEIGHTS, map_location="cpu")
    if any(k.startswith("module.") for k in sd.keys()):
        sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    i3d.load_state_dict(sd)
    i3d = i3d.to(device)
    i3d.eval()
    return i3d


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
model = load_i3d_wlasl2000(device)

In [ ]:
@torch.inference_mode()
def predict_clip(model, tensor_cthw: torch.Tensor, top_k: int = 10):
    """tensor_cthw: (C, T, H, W). Logits (B, C, T) — pool over time like test_i3d.py."""
    x = tensor_cthw.unsqueeze(0).to(device)
    per_frame_logits = model(x)
    pooled = torch.max(per_frame_logits, dim=2)[0][0]
    scores, indices = torch.topk(pooled, k=min(top_k, NUM_CLASSES))
    return indices.cpu().numpy(), scores.cpu().numpy()


def print_topk(indices, scores):
    for rank, (i, s) in enumerate(zip(indices, scores), start=1):
        gloss = idx_to_gloss.get(int(i), f"? ({i})")
        print(rank, int(i), gloss, float(s))

In [ ]:
def capture_webcam_frames(num_frames: int, cam_index: int = 0):
    cap = cv2.VideoCapture(cam_index)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open webcam {cam_index}")
    frames = []
    for _ in range(num_frames):
        ok, img = cap.read()
        if not ok:
            break
        frames.append(img)
    cap.release()
    if not frames:
        raise RuntimeError("No frames captured")
    return np.asarray(frames, dtype=np.uint8)


frames = capture_webcam_frames(NUM_FRAMES, WEBCAM_INDEX)
print("Captured", frames.shape[0], "frames", frames.shape[1:3])
proc = preprocess_frames_hwc(frames)
tensor = video_to_tensor(proc)
idx, sc = predict_clip(model, tensor, TOP_K)
print_topk(idx, sc)

## Optional: classify a video file (no webcam)
Set `VIDEO_PATH` to a short `.mp4` on this machine.

In [ ]:
def frames_from_video_path(path: Path, max_frames: int):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(path)
    frames = []
    while len(frames) < max_frames:
        ok, img = cap.read()
        if not ok:
            break
        frames.append(img)
    cap.release()
    if not frames:
        raise RuntimeError("No frames read")
    return np.asarray(frames, dtype=np.uint8)


# VIDEO_PATH = Path("/path/to/clip.mp4")
# frames = frames_from_video_path(VIDEO_PATH, NUM_FRAMES)
# proc = preprocess_frames_hwc(frames)
# tensor = video_to_tensor(proc)
# idx, sc = predict_clip(model, tensor, TOP_K)
# print_topk(idx, sc)